In [1]:
# =========================
# Enhanced Molecule Finder with Comprehensive Functional Group Analysis
# Professional version with 40+ functional groups and detailed molecular insights
# =========================

# Installation (run once in Colab):
# !pip -q install pubchempy py3Dmol rdkit-pypi ipywidgets pandas

import pubchempy as pcp
from rdkit import Chem
from rdkit.Chem import AllChem, rdMolDescriptors, Descriptors, Crippen, Lipinski
import py3Dmol
import pandas as pd
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets
from typing import List, Dict, Optional, Tuple
import warnings

In [2]:


warnings.filterwarnings('ignore')


# =========================
# COMPREHENSIVE FUNCTIONAL GROUP LIBRARY
# =========================

class FunctionalGroupLibrary:
    """Comprehensive functional group definitions with SMARTS patterns"""
    
    SMARTS = {
        # Oxygen-containing groups
        "Primary alcohol": "[CH2X4][OX2H]",
        "Secondary alcohol": "[CH1X4][OX2H]",
        "Tertiary alcohol": "[CX4][OX2H]",
        "Phenol": "c[OX2H]",
        "Ether": "[OD2]([#6])[#6]",
        "Epoxide": "C1OC1",
        "Peroxide": "[OX2][OX2]",
        
        # Carbonyl groups
        "Aldehyde": "[CX3H1](=O)[#6]",
        "Ketone": "[#6][CX3](=O)[#6]",
        "Carboxylic acid": "C(=O)[OX2H1]",
        "Ester": "C(=O)O[#6;!$(C=O)]",
        "Lactone": "C(=O)O@[C,c]",
        "Amide": "C(=O)N[#6,H]",
        "Lactam": "C(=O)N@[C,c]",
        "Carbamate": "NC(=O)O",
        "Carbonate": "OC(=O)O",
        "Acyl halide": "C(=O)[F,Cl,Br,I]",
        "Acid anhydride": "C(=O)OC(=O)",
        "Imide": "C(=O)NC(=O)",
        "Urea": "NC(=O)N",
        "Thiourea": "NC(=S)N",
        
        # Nitrogen-containing groups
        "Primary amine": "[NX3;H2;!$(NC=O)]",
        "Secondary amine": "[NX3;H1;!$(NC=O);!$(N-N)]",
        "Tertiary amine": "[NX3;H0;!$(NC=O);!$(N-N);!$(N=*)]",
        "Quaternary ammonium": "[NX4+]",
        "Aniline": "c[NX3]",
        "Imine": "[CX3]=[NX2]",
        "Hydrazone": "[NX3][NX2]=[CX3]",
        "Oxime": "[CX3]=[NX2][OX2H]",
        "Nitrile": "[NX1]#[CX2]",
        "Isonitrile": "[CX1-]#[NX2+]",
        "Nitro": "[$([NX3](=O)=O),$([NX3+](=O)[O-])][!#8]",
        "Nitroso": "[NX2]=O",
        "Azide": "N=[N+]=[N-]",
        "Azo": "N=N",
        "Hydrazine": "[NX3][NX3]",
        "Diazonium": "[N+]#N",
        
        # Sulfur-containing groups
        "Thiol": "[SX2H]",
        "Sulfide": "[#6][SX2][#6]",
        "Disulfide": "[SX2][SX2]",
        "Sulfoxide": "[SX3](=O)[#6]",
        "Sulfone": "[SX4](=O)(=O)[#6]",
        "Sulfonic acid": "[SX4](=O)(=O)[OX2H]",
        "Sulfonamide": "[SX4](=O)(=O)[NX3]",
        "Sulfonate ester": "[SX4](=O)(=O)O[#6]",
        "Thioester": "C(=O)S[#6]",
        "Thiocarbonyl": "[CX3]=[SX1]",
        "Isothiocyanate": "N=C=S",
        
        # Phosphorus-containing groups
        "Phosphate": "[PX4](=O)([OX2])[OX2]",
        "Phosphonate": "[PX4](=O)([OX2])[#6]",
        "Phosphine": "[PX3]",
        "Phosphine oxide": "[PX4](=O)",
        
        # Halogen-containing groups
        "Fluoride": "[F][CX4]",
        "Chloride": "[Cl][CX4]",
        "Bromide": "[Br][CX4]",
        "Iodide": "[I][CX4]",
        "Aryl halide": "[F,Cl,Br,I]c",
        
        # Ring systems and aromatic groups
        "Benzene ring": "c1ccccc1",
        "Pyridine": "n1ccccc1",
        "Pyrrole": "[nH]1cccc1",
        "Furan": "o1cccc1",
        "Thiophene": "s1cccc1",
        "Imidazole": "n1c[nH]cc1",
        "Oxazole": "n1cocc1",
        "Thiazole": "n1cscc1",
        "Indole": "c1ccc2c(c1)[nH]cc2",
        "Quinoline": "c1ccc2ncccc2c1",
        "Isoquinoline": "c1cnc2ccccc2c1",
        
        # Other important groups
        "Alkene": "C=C",
        "Alkyne": "C#C",
        "Allene": "C=C=C",
        "Acetal": "[CX4]([OX2][#6])([OX2][#6])[#6]",
        "Hemiacetal": "[CX4]([OX2][#6])([OX2H])[#6]",
        "Ketal": "[CX4]([OX2][#6])([OX2][#6])([#6])[#6]",
        "Cyanohydrin": "[CX4]([OX2H])[CX2]#[NX1]",
    }
    
    ATTRIBUTES = {
        # Oxygen groups
        "Primary alcohol": "Polar; H-bond donor/acceptor; metabolic target for oxidation to aldehyde/acid; good solubility",
        "Secondary alcohol": "Polar; H-bond donor/acceptor; metabolic target for oxidation to ketone; moderate solubility",
        "Tertiary alcohol": "Polar; H-bond donor/acceptor; resistant to oxidation; prone to dehydration",
        "Phenol": "Weak acid (pKa ~10); H-bond donor; prone to glucuronidation/sulfation; antioxidant potential",
        "Ether": "H-bond acceptor only; moderate polarity; metabolically stable (but O-dealkylation possible); lipophilic",
        "Epoxide": "Highly reactive; electrophilic; potential toxicity (DNA binding); metabolic intermediate",
        "Peroxide": "Oxidizing agent; unstable; potential toxicity; antimalarial activity (artemisinin)",
        
        # Carbonyl groups
        "Aldehyde": "Reactive electrophile; oxidation → carboxylic acid; reduction → alcohol; Schiff base formation",
        "Ketone": "Moderate reactivity; reduction → alcohol; imine formation; metabolically stable",
        "Carboxylic acid": "Strong H-bond donor; acidic (pKa 3-5); ionized at physiological pH; ↑solubility, ↓permeability",
        "Ester": "Hydrolyzable by esterases; prodrug strategy; lipophilic; metabolic soft spot",
        "Lactone": "Cyclic ester; ring-opening hydrolysis; found in many natural products; moderate stability",
        "Amide": "Resonance-stabilized; H-bond donor/acceptor; peptide bond equivalent; very stable in vivo",
        "Lactam": "Cyclic amide; β-lactam = antibiotic pharmacophore; stable but hydrolyzable under stress",
        "Carbamate": "Ester + amide hybrid; moderately stable; prodrug linker; insecticide mechanism",
        "Carbonate": "Ester of carbonic acid; hydrolyzable; prodrug strategy; moderately stable",
        "Acyl halide": "Highly reactive; acylating agent; not drug-like; synthetic intermediate only",
        "Acid anhydride": "Reactive; acylating agent; hydrolysis → carboxylic acids; not typically in drugs",
        "Imide": "Weak acid; H-bond donor/acceptor; stable; found in phthalimides and barbiturates",
        "Urea": "H-bond donor/acceptor; polar; metabolically stable; found in many drugs",
        "Thiourea": "Similar to urea but sulfur analog; coordination chemistry; potential toxicity",
        
        # Nitrogen groups
        "Primary amine": "Basic (pKa 9-11); ionized at pH 7.4; H-bond donor/acceptor; ↑solubility; salt formation",
        "Secondary amine": "Basic (pKa 9-11); ionized at pH 7.4; H-bond donor/acceptor; metabolic N-dealkylation",
        "Tertiary amine": "Basic (pKa 9-11); ionized at pH 7.4; H-bond acceptor only; lipophilic when neutral",
        "Quaternary ammonium": "Permanently charged; very hydrophilic; poor membrane permeability; muscarinic antagonists",
        "Aniline": "Weakly basic (pKa ~4-5); aromatic amine; oxidative metabolism risk; potential toxicity alert",
        "Imine": "Hydrolyzable; electrophilic carbon; Schiff base; reversible covalent bonding",
        "Hydrazone": "Similar to imine; pH-sensitive; drug-linker in ADCs; metabolically labile",
        "Oxime": "More stable than imine; H-bond donor; nerve agent antidote mechanism",
        "Nitrile": "Weakly polar; linear geometry; metabolic hydrolysis → carboxylic acid; bioisostere",
        "Isonitrile": "Rare; reactive; unpleasant odor; synthetic utility",
        "Nitro": "Strong electron-withdrawing; potential genotoxicity; nitro reduction in hypoxia; explosophore",
        "Nitroso": "Reactive; N-nitrosamine = carcinogen alert; DNA alkylation",
        "Azide": "Explosive hazard; bioorthogonal click chemistry; metabolic reduction possible",
        "Azo": "Chromophore; metabolic cleavage; prodrug strategy; textile dyes",
        "Hydrazine": "Reactive nucleophile; potential toxicity; MAO inhibitor scaffold; carcinogen alert",
        "Diazonium": "Highly reactive; synthetic intermediate; not stable in biological systems",
        
        # Sulfur groups
        "Thiol": "Nucleophilic; redox-active; disulfide formation; strong H-bond donor; malodorous",
        "Sulfide": "Moderate polarity; metabolic S-oxidation → sulfoxide → sulfone; nucleophilic",
        "Disulfide": "Redox-active; cleavable bond; protein structure stabilization; prodrug strategy",
        "Sulfoxide": "Polar; H-bond acceptor; chiral center; further oxidation → sulfone",
        "Sulfone": "Very polar; metabolically stable; H-bond acceptor; found in many drugs",
        "Sulfonic acid": "Strong acid (pKa ~1); ionized; very hydrophilic; poor permeability",
        "Sulfonamide": "Weakly acidic; H-bond donor/acceptor; antibacterial pharmacophore; metabolically stable",
        "Sulfonate ester": "Good leaving group; alkylating agent; potential genotoxicity",
        "Thioester": "More reactive than ester; bioisostere; acetyl-CoA analog; hydrolyzable",
        "Thiocarbonyl": "Rare in drugs; more reactive than carbonyl; bioisostere",
        "Isothiocyanate": "Electrophilic; protein modification; mustard oils; potential toxicity",
        
        # Phosphorus groups
        "Phosphate": "Highly polar; anionic; prodrug strategy; metabolic energy carrier; poor permeability",
        "Phosphonate": "Bioisostere of phosphate; more stable; transition state analog; enzyme inhibitor",
        "Phosphine": "Nucleophilic; ligand in catalysis; air-sensitive; rare in drugs",
        "Phosphine oxide": "Polar; H-bond acceptor; metabolically stable; bioisostere",
        
        # Halogens
        "Fluoride": "↑lipophilicity; ↑metabolic stability; strong C-F bond; bioisostere for H/OH",
        "Chloride": "↑lipophilicity; moderate stability; larger than F; bioisostere",
        "Bromide": "↑lipophilicity; good leaving group; larger; environmental persistence concern",
        "Iodide": "Very lipophilic; thyroid activity; radioiodine therapy; heavy atom",
        "Aryl halide": "Metabolically stable; ↑lipophilicity; ↑potency; common in CNS drugs",
        
        # Aromatic rings
        "Benzene ring": "Hydrophobic; π-π stacking; metabolic hydroxylation; flat geometry",
        "Pyridine": "Weakly basic (pKa ~5); H-bond acceptor; bioisostere of benzene; metabolic N-oxidation",
        "Pyrrole": "Aromatic; H-bond donor; metabolic oxidation; found in porphyrins",
        "Furan": "Aromatic O-heterocycle; metabolic opening → toxic aldehydes; found in natural products",
        "Thiophene": "Aromatic S-heterocycle; bioisostere of benzene; metabolic S-oxidation; stable",
        "Imidazole": "Aromatic; basic; H-bond donor/acceptor; histidine analog; found in many drugs",
        "Oxazole": "Aromatic; electron-rich; found in natural products; moderate stability",
        "Thiazole": "Aromatic; thiamin structure; found in many drugs; metabolically stable",
        "Indole": "Aromatic; H-bond donor; tryptophan analog; neurotransmitter precursor",
        "Quinoline": "Aromatic; weakly basic; antimalarial pharmacophore; flat structure",
        "Isoquinoline": "Aromatic; weakly basic; alkaloid structure; neurotoxic potential",
        
        # Other groups
        "Alkene": "Planar; π-bond; metabolic epoxidation; geometric isomerism; moderate reactivity",
        "Alkyne": "Linear; π-bonds; metabolic activation; bioorthogonal click chemistry; rare in drugs",
        "Allene": "Cumulated double bonds; chiral; rare; reactive; synthetic intermediate",
        "Acetal": "Acid-labile; protecting group; prodrug linker; stable at neutral/basic pH",
        "Hemiacetal": "Equilibrium with aldehyde/ketone; sugar structure; reactive",
        "Ketal": "Similar to acetal; acid-labile; protecting group; steroid structure",
        "Cyanohydrin": "Cyanide precursor; toxic; enzyme inhibitor; unstable",
    }


# =========================
# MOLECULAR PROPERTY ANALYSIS
# =========================

class MolecularPropertyCalculator:
    """Comprehensive molecular property calculations and predictions"""
    
    @staticmethod
    def calculate_properties(mol: Chem.Mol) -> Dict[str, float]:
        """Calculate comprehensive molecular descriptors"""
        return {
            # Basic properties
            "molecular_weight": Descriptors.MolWt(mol),
            "exact_mass": Descriptors.ExactMolWt(mol),
            "heavy_atom_count": Descriptors.HeavyAtomCount(mol),
            
            # Lipophilicity
            "logp": Crippen.MolLogP(mol),
            "molar_refractivity": Crippen.MolMR(mol),
            
            # H-bonding
            "hbd": Lipinski.NumHDonors(mol),
            "hba": Lipinski.NumHAcceptors(mol),
            
            # Polarity
            "tpsa": rdMolDescriptors.CalcTPSA(mol),
            
            # Rotatable bonds (flexibility)
            "rotatable_bonds": Descriptors.NumRotatableBonds(mol),
            
            # Rings
            "aromatic_rings": rdMolDescriptors.CalcNumAromaticRings(mol),
            "aliphatic_rings": rdMolDescriptors.CalcNumAliphaticRings(mol),
            "ring_count": rdMolDescriptors.CalcNumRings(mol),
            
            # Saturation
            "fraction_csp3": rdMolDescriptors.CalcFractionCSP3(mol),
            
            # Complexity
            "num_stereocenters": rdMolDescriptors.CalcNumAtomStereoCenters(mol),
        }
    
    @staticmethod
    def assess_drug_likeness(props: Dict[str, float]) -> Dict[str, any]:
        """Assess drug-likeness using multiple rules"""
        
        # Lipinski's Rule of Five
        lipinski_violations = 0
        lipinski_reasons = []
        
        if props["molecular_weight"] > 500:
            lipinski_violations += 1
            lipinski_reasons.append(f"MW > 500 ({props['molecular_weight']:.1f})")
        if props["logp"] > 5:
            lipinski_violations += 1
            lipinski_reasons.append(f"LogP > 5 ({props['logp']:.2f})")
        if props["hbd"] > 5:
            lipinski_violations += 1
            lipinski_reasons.append(f"HBD > 5 ({props['hbd']})")
        if props["hba"] > 10:
            lipinski_violations += 1
            lipinski_reasons.append(f"HBA > 10 ({props['hba']})")
        
        lipinski_pass = lipinski_violations <= 1
        
        # Veber's rules (oral bioavailability)
        veber_pass = (props["rotatable_bonds"] <= 10 and props["tpsa"] <= 140)
        veber_reasons = []
        if props["rotatable_bonds"] > 10:
            veber_reasons.append(f"RotBonds > 10 ({props['rotatable_bonds']})")
        if props["tpsa"] > 140:
            veber_reasons.append(f"tPSA > 140 ({props['tpsa']:.1f})")
        
        # Ghose filter
        ghose_pass = (160 <= props["molecular_weight"] <= 480 and
                      -0.4 <= props["logp"] <= 5.6 and
                      40 <= props["molar_refractivity"] <= 130 and
                      20 <= props["heavy_atom_count"] <= 70)
        
        # Lead-likeness
        lead_like = (250 <= props["molecular_weight"] <= 350 and
                     props["logp"] <= 3.5 and
                     props["rotatable_bonds"] <= 7)
        
        # CNS penetration (Pajouhesh and Lenz)
        cns_mpo_score = 0  # Simplified estimation
        if props["molecular_weight"] <= 360: cns_mpo_score += 1
        if props["logp"] <= 3: cns_mpo_score += 1
        if props["hbd"] <= 0.5: cns_mpo_score += 1
        if props["tpsa"] <= 60: cns_mpo_score += 1
        
        return {
            "lipinski": {
                "pass": lipinski_pass,
                "violations": lipinski_violations,
                "reasons": lipinski_reasons
            },
            "veber": {
                "pass": veber_pass,
                "reasons": veber_reasons
            },
            "ghose": ghose_pass,
            "lead_like": lead_like,
            "cns_penetration": "Likely" if cns_mpo_score >= 3 else "Unlikely" if cns_mpo_score <= 1 else "Possible"
        }
    
    @staticmethod
    def determine_chemical_class(fg_status: Dict[str, bool], props: Dict[str, float]) -> str:
        """Determine overall chemical classification"""
        classes = []
        
        # Aromatic vs aliphatic
        if fg_status.get("Benzene ring", False) or any(fg_status.get(ring, False) 
            for ring in ["Pyridine", "Pyrrole", "Furan", "Thiophene", "Imidazole", "Indole"]):
            classes.append("Aromatic")
        if props.get("fraction_csp3", 0) > 0.5:
            classes.append("Aliphatic")
        
        # Heterocyclic
        if any(fg_status.get(ring, False) for ring in 
               ["Pyridine", "Pyrrole", "Furan", "Thiophene", "Imidazole", "Indole", "Quinoline"]):
            classes.append("Heterocyclic")
        
        # Specific functional class
        if fg_status.get("Primary amine", False) or fg_status.get("Secondary amine", False):
            classes.append("Amine")
        if fg_status.get("Carboxylic acid", False):
            classes.append("Carboxylic acid")
        if fg_status.get("Phenol", False):
            classes.append("Phenolic")
        if fg_status.get("Amide", False):
            classes.append("Amide")
        
        return " / ".join(classes) if classes else "Simple organic"
    
    @staticmethod
    def determine_acidity(fg_status: Dict[str, bool]) -> str:
        """Determine overall acidity/basicity with nuance"""
        acidic_groups = ["Carboxylic acid", "Phenol", "Sulfonic acid", "Sulfonamide"]
        basic_groups = ["Primary amine", "Secondary amine", "Tertiary amine", "Pyridine", "Imidazole"]
        
        has_acidic = any(fg_status.get(g, False) for g in acidic_groups)
        has_basic = any(fg_status.get(g, False) for g in basic_groups)
        
        if has_acidic and has_basic:
            return "Amphoteric (zwitterionic at some pH)"
        elif has_acidic:
            # Determine strength
            if fg_status.get("Sulfonic acid", False):
                return "Strongly acidic (pKa < 2)"
            elif fg_status.get("Carboxylic acid", False):
                return "Weakly acidic (pKa 3-5)"
            elif fg_status.get("Phenol", False):
                return "Very weakly acidic (pKa ~10)"
            else:
                return "Acidic"
        elif has_basic:
            if fg_status.get("Quaternary ammonium", False):
                return "Permanently cationic"
            elif fg_status.get("Aniline", False):
                return "Very weakly basic (pKa ~4-5)"
            else:
                return "Basic (pKa 9-11, ionized at pH 7.4)"
        else:
            return "Neutral"
    
    @staticmethod
    def generate_comprehensive_summary(mol: Chem.Mol, fg_status: Dict[str, bool]) -> str:
        """Generate detailed molecular summary"""
        props = MolecularPropertyCalculator.calculate_properties(mol)
        drug_like = MolecularPropertyCalculator.assess_drug_likeness(props)
        chem_class = MolecularPropertyCalculator.determine_chemical_class(fg_status, props)
        acidity = MolecularPropertyCalculator.determine_acidity(fg_status)
        
        # Polarity assessment
        if props["tpsa"] > 140:
            polarity = "Highly polar"
        elif props["tpsa"] > 75:
            polarity = "Moderately polar"
        else:
            polarity = "Low polarity"
        
        # Lipophilicity
        if props["logp"] > 5:
            lipophilicity = "Highly lipophilic"
        elif props["logp"] > 3:
            lipophilicity = "Moderately lipophilic"
        elif props["logp"] > 0:
            lipophilicity = "Balanced lipophilicity"
        else:
            lipophilicity = "Hydrophilic"
        
        # Protic character
        protic = "Protic" if props["hbd"] >= 1 else "Aprotic"
        
        # Flexibility
        if props["rotatable_bonds"] > 10:
            flexibility = "Highly flexible"
        elif props["rotatable_bonds"] > 5:
            flexibility = "Moderately flexible"
        else:
            flexibility = "Rigid"
        
        # Build summary
        summary_parts = [
            f"**Chemical Class:** {chem_class}",
            f"**Character:** {polarity}, {protic}, {acidity}, {lipophilicity}",
            f"**Flexibility:** {flexibility} ({props['rotatable_bonds']} rotatable bonds)",
            f"**Saturation:** {props['fraction_csp3']*100:.0f}% sp³ carbons",
            "",
            f"**Molecular Weight:** {props['molecular_weight']:.2f} g/mol",
            f"**LogP:** {props['logp']:.2f} (lipophilicity)",
            f"**tPSA:** {props['tpsa']:.1f} Ų (polar surface area)",
            f"**H-bond Donors:** {props['hbd']} | **Acceptors:** {props['hba']}",
            f"**Aromatic Rings:** {props['aromatic_rings']} | **Total Rings:** {props['ring_count']}",
            ""
        ]
        
        # Drug-likeness assessment
        summary_parts.append("**Drug-Likeness Assessment:**")
        
        if drug_like["lipinski"]["pass"]:
            summary_parts.append(f"✓ **Lipinski's Rule of Five:** PASS ({drug_like['lipinski']['violations']} violations)")
        else:
            violations = ", ".join(drug_like["lipinski"]["reasons"])
            summary_parts.append(f"✗ **Lipinski's Rule of Five:** FAIL ({violations})")
        
        if drug_like["veber"]["pass"]:
            summary_parts.append("✓ **Veber's Rules (Oral bioavailability):** PASS")
        else:
            reasons = ", ".join(drug_like["veber"]["reasons"])
            summary_parts.append(f"✗ **Veber's Rules:** FAIL ({reasons})")
        
        summary_parts.append(f"{'✓' if drug_like['ghose'] else '✗'} **Ghose Filter:** {'PASS' if drug_like['ghose'] else 'FAIL'}")
        summary_parts.append(f"{'✓' if drug_like['lead_like'] else '✗'} **Lead-likeness:** {'YES' if drug_like['lead_like'] else 'NO'}")
        summary_parts.append(f"**CNS Penetration:** {drug_like['cns_penetration']}")
        
        return "\n".join(summary_parts)


# =========================
# FUNCTIONAL GROUP ANALYSIS
# =========================

class FunctionalGroupAnalyzer:
    """Handles functional group detection and analysis"""
    
    @staticmethod
    def detect(mol: Chem.Mol) -> Dict[str, bool]:
        """Detect functional groups in a molecule using SMARTS patterns"""
        results = {}
        for name, smarts in FunctionalGroupLibrary.SMARTS.items():
            try:
                pattern = Chem.MolFromSmarts(smarts)
                if pattern is not None:
                    results[name] = bool(mol.HasSubstructMatch(pattern))
                else:
                    results[name] = False
            except Exception:
                results[name] = False
        return results
    
    @staticmethod
    def create_dataframe(fg_status: Dict[str, bool]) -> pd.DataFrame:
        """Create a formatted DataFrame for functional group display"""
        rows = []
        for fg_name, status in fg_status.items():
            if status:  # Only show detected groups
                attribute = FunctionalGroupLibrary.ATTRIBUTES.get(fg_name, "")
                rows.append({
                    "Functional Group": fg_name,
                    "Status": "✓ Present",
                    "Pharmacological/Chemical Attributes": attribute
                })
        
        # If no functional groups detected
        if not rows:
            rows.append({
                "Functional Group": "Simple hydrocarbon",
                "Status": "—",
                "Pharmacological/Chemical Attributes": "No heteroatoms or special functional groups detected"
            })
        
        return pd.DataFrame(rows)


# =========================
# 3D STRUCTURE GENERATION
# =========================

class Structure3DGenerator:
    """Handles 3D structure generation and visualization"""
    
    @staticmethod
    def generate_3d(mol_2d: Chem.Mol) -> Chem.Mol:
        """Generate 3D coordinates for a molecule"""
        mol_3d = Chem.AddHs(mol_2d)
        
        params = AllChem.ETKDG()
        params.randomSeed = 42
        
        result = AllChem.EmbedMolecule(mol_3d, params)
        
        if result != 0:
            params.useRandomCoords = True
            result = AllChem.EmbedMolecule(mol_3d, params)
            
            if result != 0:
                raise ValueError("Failed to generate 3D coordinates for this molecule.")
        
        try:
            AllChem.UFFOptimizeMolecule(mol_3d)
        except Exception:
            pass
        
        return mol_3d
    
    @staticmethod
    def visualize(mol_3d: Chem.Mol, width: int = 520, height: int = 420):
        """Create 3D visualization using py3Dmol"""
        mol_block = Chem.MolToMolBlock(mol_3d)
        viewer = py3Dmol.view(width=width, height=height)
        viewer.addModel(mol_block, "mol")
        viewer.setStyle({"stick": {}})
        viewer.zoomTo()
        return viewer.show()


# =========================
# PUBCHEM INTERFACE
# =========================

class PubChemInterface:
    """Handles all PubChem API interactions"""
    
    VALID_SEARCH_TYPES = {'formula', 'smiles', 'inchi', 'name', 'cid'}
    
    @staticmethod
    def search(identifier: str, search_type: str, max_results: int = 25) -> List:
        """Search PubChem for compounds"""
        identifier = identifier.strip()
        if not identifier:
            return []
        
        search_type = search_type.lower()
        if search_type not in PubChemInterface.VALID_SEARCH_TYPES:
            raise ValueError(f"Invalid search type: {search_type}")
        
        try:
            if search_type == 'cid':
                cid = int(identifier)
                compounds = pcp.get_compounds(cid, 'cid')
            else:
                compounds = pcp.get_compounds(identifier, search_type)
            
            compounds = compounds or []
            return compounds[:max_results]
            
        except ValueError as e:
            if search_type == 'cid':
                raise ValueError(f"CID must be an integer (e.g., 962). Got: {identifier}") from e
            raise
        except Exception as e:
            raise RuntimeError(f"PubChem search failed: {str(e)}") from e
    
    @staticmethod
    def get_smiles(compound) -> Optional[str]:
        """Extract SMILES from PubChem compound"""
        smiles = getattr(compound, 'connectivity_smiles', None)
        if not smiles:
            smiles = getattr(compound, 'canonical_smiles', None)
        return smiles
    
    @staticmethod
    def format_label(compound) -> str:
        """Create a formatted label for display"""
        cid = getattr(compound, 'cid', 'N/A')
        name = getattr(compound, 'iupac_name', None) or 'N/A'
        smiles = PubChemInterface.get_smiles(compound) or 'N/A'
        
        if isinstance(smiles, str) and len(smiles) > 50:
            smiles = smiles[:50] + "…"
        
        return f"CID {cid} | {name} | {smiles}"


# =========================
# USER INTERFACE
# =========================

class MoleculeFinderApp:
    """Main application class managing the user interface"""
    
    def __init__(self):
        self.compounds = []
        self.setup_widgets()
        self.setup_callbacks()
    
    def setup_widgets(self):
        """Initialize all UI widgets"""
        self.identifier_input = widgets.Text(
            value='C1=CC=C(C=C1)O',
            description='Query:',
            placeholder='e.g., H2O, CCO, InChI=1S/H2O/h1H2, ethanol, 962',
            layout=widgets.Layout(width='520px')
        )
        
        self.search_type_dropdown = widgets.Dropdown(
            options=[
                ('Formula', 'formula'),
                ('Canonical SMILES', 'smiles'),
                ('InChI', 'inchi'),
                ('Name', 'name'),
                ('CID', 'cid'),
            ],
            value='smiles',
            description='Type:',
            layout=widgets.Layout(width='260px')
        )
        
        self.search_button = widgets.Button(
            description='Search PubChem',
            button_style='primary',
            icon='search'
        )
        
        self.approve_button = widgets.Button(
            description='Analyze Molecule',
            button_style='success',
            icon='check',
            disabled=True
        )
        
        self.results_dropdown = widgets.Dropdown(
            options=[],
            description='Results:',
            layout=widgets.Layout(width='900px')
        )
        
        self.status_output = widgets.Output()
        self.preview_output = widgets.Output()
        self.final_output = widgets.Output()
    
    def setup_callbacks(self):
        """Connect event handlers to widgets"""
        self.search_button.on_click(self.handle_search)
        self.approve_button.on_click(self.handle_approve)
        self.results_dropdown.observe(self.handle_selection_change, names='value')
    
    def handle_search(self, button):
        """Handle search button click"""
        with self.status_output:
            clear_output()
            print("🔍 Searching PubChem...")
        
        with self.preview_output:
            clear_output()
        
        with self.final_output:
            clear_output()
        
        identifier = self.identifier_input.value.strip()
        search_type = self.search_type_dropdown.value
        
        if not identifier:
            with self.status_output:
                clear_output()
                print("⚠️ Please enter a search query.")
            self.results_dropdown.options = []
            self.approve_button.disabled = True
            return
        
        try:
            self.compounds = PubChemInterface.search(identifier, search_type)
            
            if not self.compounds:
                with self.status_output:
                    clear_output()
                    print(f"❌ No compounds found for {search_type} = '{identifier}'")
                self.results_dropdown.options = []
                self.approve_button.disabled = True
                return
            
            options = [
                (PubChemInterface.format_label(compound), idx)
                for idx, compound in enumerate(self.compounds)
            ]
            self.results_dropdown.options = options
            self.results_dropdown.value = 0
            self.approve_button.disabled = False
            
            with self.status_output:
                clear_output()
                print(f"✓ Found {len(self.compounds)} result(s). Select one and click 'Analyze Molecule'.")
            
            self.preview_compound(self.compounds[0])
            
        except Exception as e:
            with self.status_output:
                clear_output()
                print(f"❌ Error: {str(e)}")
            self.results_dropdown.options = []
            self.approve_button.disabled = True
    
    def handle_selection_change(self, change):
        """Handle dropdown selection change"""
        if change['name'] == 'value' and self.compounds:
            idx = change['new']
            if idx is not None:
                self.preview_compound(self.compounds[idx])
    
    def handle_approve(self, button):
        """Handle approve button click"""
        with self.final_output:
            clear_output()
        
        if not self.compounds:
            with self.status_output:
                clear_output()
                print("⚠️ No search results available. Please search first.")
            return
        
        idx = self.results_dropdown.value
        compound = self.compounds[idx]
        
        with self.final_output:
            try:
                self.display_full_analysis(compound)
            except Exception as e:
                display(HTML(f"""
                    <div style='background-color: #fee; border: 1px solid #c33; padding: 10px; border-radius: 5px;'>
                        <strong>❌ Error during analysis:</strong> {str(e)}
                    </div>
                """))
    
    def preview_compound(self, compound):
        """Display preview of selected compound"""
        with self.preview_output:
            clear_output()
            try:
                cid = getattr(compound, 'cid', 'N/A')
                name = getattr(compound, 'iupac_name', None) or 'N/A'
                smiles = PubChemInterface.get_smiles(compound)
                
                if not smiles:
                    display(HTML("""
                        <div style='background-color: #ffc; border: 1px solid #cc6; padding: 10px; border-radius: 5px;'>
                            <strong>⚠️ Warning:</strong> No SMILES available for this compound.
                        </div>
                    """))
                    return
                
                mol = Chem.MolFromSmiles(smiles)
                if mol is None:
                    display(HTML("""
                        <div style='background-color: #ffc; border: 1px solid #cc6; padding: 10px; border-radius: 5px;'>
                            <strong>⚠️ Warning:</strong> RDKit could not parse SMILES.
                        </div>
                    """))
                    return
                
                props = MolecularPropertyCalculator.calculate_properties(mol)
                
                display(HTML(f"""
                    <div style='background-color: #f0f8ff; border: 1px solid #4682b4; padding: 15px; border-radius: 5px; margin-bottom: 10px;'>
                        <h3 style='margin-top: 0; color: #4682b4;'>📋 Quick Preview</h3>
                        <table style='width: 100%; border-collapse: collapse;'>
                            <tr><td style='padding: 5px;'><strong>CID:</strong></td><td style='padding: 5px;'>{cid}</td></tr>
                            <tr><td style='padding: 5px;'><strong>Name:</strong></td><td style='padding: 5px;'>{name}</td></tr>
                            <tr><td style='padding: 5px;'><strong>SMILES:</strong></td><td style='padding: 5px; font-family: monospace;'>{smiles}</td></tr>
                            <tr><td style='padding: 5px;'><strong>MW:</strong></td><td style='padding: 5px;'>{props['molecular_weight']:.2f} g/mol</td></tr>
                        </table>
                    </div>
                """))
                
                mol_3d = Structure3DGenerator.generate_3d(mol)
                Structure3DGenerator.visualize(mol_3d)
                
            except Exception as e:
                display(HTML(f"""
                    <div style='background-color: #fee; border: 1px solid #c33; padding: 10px; border-radius: 5px;'>
                        <strong>❌ Preview failed:</strong> {str(e)}
                    </div>
                """))
    
    def display_full_analysis(self, compound):
        """Display complete molecular analysis"""
        cid = getattr(compound, 'cid', 'N/A')
        name = getattr(compound, 'iupac_name', None) or 'N/A'
        smiles = PubChemInterface.get_smiles(compound)
        
        if not smiles:
            raise ValueError("Selected compound has no SMILES data in PubChem.")
        
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            raise ValueError("RDKit could not parse the SMILES string.")
        
        # Perform analysis
        fg_status = FunctionalGroupAnalyzer.detect(mol)
        summary = MolecularPropertyCalculator.generate_comprehensive_summary(mol, fg_status)
        fg_dataframe = FunctionalGroupAnalyzer.create_dataframe(fg_status)
        
        # Display header
        display(HTML(f"""
            <div style='background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; padding: 25px; border-radius: 10px; margin-bottom: 20px; box-shadow: 0 4px 6px rgba(0,0,0,0.1);'>
                <h1 style='margin: 0 0 10px 0; font-size: 28px;'>🧬 {name}</h1>
                <p style='margin: 5px 0; font-size: 16px; opacity: 0.9;'><strong>PubChem CID:</strong> {cid}</p>
                <div style='background-color: rgba(255,255,255,0.15); padding: 12px; border-radius: 6px; margin-top: 12px;'>
                    <p style='margin: 0; font-family: "Courier New", monospace; font-size: 14px; word-break: break-all;'><strong>SMILES:</strong> {smiles}</p>
                </div>
            </div>
        """))
        
        # Display comprehensive summary
        display(HTML(f"""
            <div style='background-color: #f8f9fa; border-left: 5px solid #4682b4; padding: 20px; margin-bottom: 25px; border-radius: 5px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);'>
                <h2 style='margin-top: 0; color: #2c3e50; border-bottom: 2px solid #4682b4; padding-bottom: 10px;'>📊 Comprehensive Molecular Analysis</h2>
                <div style='line-height: 1.8; color: #34495e; white-space: pre-line;'>{summary}</div>
            </div>
        """))
        
        # Display functional groups table
        display(HTML("""
            <div style='margin-bottom: 20px;'>
                <h2 style='color: #2c3e50; border-bottom: 2px solid #e74c3c; padding-bottom: 10px;'>🔬 Detected Functional Groups & Their Properties</h2>
                <p style='color: #7f8c8d; font-style: italic; margin-top: 10px;'>Only showing functional groups present in this molecule</p>
            </div>
        """))
        
        # Style the dataframe with better formatting
        styled_df = fg_dataframe.style.set_properties(**{
            'text-align': 'left',
            'padding': '12px',
            'font-size': '14px',
        }).set_table_styles([
            {'selector': 'th', 'props': [
                ('background-color', '#34495e'),
                ('color', 'white'),
                ('font-weight', 'bold'),
                ('padding', '15px'),
                ('text-align', 'left'),
                ('font-size', '15px')
            ]},
            {'selector': 'td', 'props': [
                ('border', '1px solid #ddd'),
                ('vertical-align', 'top')
            ]},
            {'selector': 'tr:nth-child(even)', 'props': [
                ('background-color', '#f9f9f9')
            ]},
            {'selector': 'tr:hover', 'props': [
                ('background-color', '#e3f2fd'),
                ('transition', 'background-color 0.3s')
            ]},
            {'selector': 'table', 'props': [
                ('border-collapse', 'collapse'),
                ('width', '100%'),
                ('box-shadow', '0 2px 8px rgba(0,0,0,0.1)'),
                ('border-radius', '8px'),
                ('overflow', 'hidden')
            ]}
        ]).set_table_attributes('class="dataframe"')
        
        display(styled_df)
        
        # Display 3D structure
        display(HTML("""
            <div style='margin-top: 30px; margin-bottom: 15px;'>
                <h2 style='color: #2c3e50; border-bottom: 2px solid #27ae60; padding-bottom: 10px;'>🔍 Interactive 3D Structure</h2>
                <p style='color: #7f8c8d; font-style: italic;'>Click and drag to rotate • Scroll to zoom</p>
            </div>
        """))
        
        mol_3d = Structure3DGenerator.generate_3d(mol)
        Structure3DGenerator.visualize(mol_3d)
    
    def run(self):
        """Display the application interface"""
        ui = widgets.VBox([
            widgets.HTML("""
                <div style='background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); padding: 30px; border-radius: 15px; margin-bottom: 25px; box-shadow: 0 8px 16px rgba(0,0,0,0.2);'>
                    <h1 style='color: white; margin: 0; text-align: center; font-size: 36px; text-shadow: 2px 2px 4px rgba(0,0,0,0.3);'>🧪 Advanced Molecule Analyzer</h1>
                    <p style='color: white; margin: 15px 0 0 0; text-align: center; font-size: 18px; opacity: 0.95;'>Search PubChem • Detect 70+ Functional Groups • Drug-Likeness Assessment • 3D Visualization</p>
                </div>
            """),
            widgets.HBox([
                self.identifier_input,
                self.search_type_dropdown,
                self.search_button,
                self.approve_button
            ]),
            self.status_output,
            self.results_dropdown,
            self.preview_output,
            widgets.HTML("<hr style='margin: 30px 0; border: none; border-top: 3px solid #ddd;'>"),
            self.final_output
        ])
        
        display(ui)


# =========================
# MAIN ENTRY POINT
# =========================

def main():
    """Launch the Enhanced Molecule Analyzer"""
    app = MoleculeFinderApp()
    app.run()


# Run the application
main()
